# Comparative analysis

This notebook only reads completed per-video summary artifacts. It never loads classifiers or reruns video analysis. Each workflow produces a validation table before comparisons. Explicit metadata (`video_path, group, treatment, animal, subject, region, well, day`) overrides folder-name inference. For treatment folders organized as `treatment/animal/video`, `animal` is inferred automatically from the parent folder.

In [21]:
from pathlib import Path
import sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'gcamp_analysis').is_dir() else WORKING_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from gcamp_analysis.experiments.comparative import (
    run_hierarchical_comparison,
    run_longitudinal_comparison,
    run_treatment_comparison,
    save_comparison_tables,
)
from gcamp_analysis.reporting import save_comparisons

## Longitudinal groups

With `ALIGN=False`, this compares whole-video descriptive statistics across days and makes no cell-identity claim. `ALIGN=True` adds spatial registration, ROI matching, and cell/group tracking while retaining the descriptive tables.

In [15]:
LONGITUDINAL_GROUPS = {
    # 'region_1': Path(r'C:\\path\\to\\region_1'),
}
LONGITUDINAL_METADATA = None  # optional CSV/XLSX/Parquet path or DataFrame
ALIGN = False
LONGITUDINAL_OUTPUT = PROJECT_ROOT / 'comparison_outputs' / 'longitudinal'

In [16]:
longitudinal_result = None
if LONGITUDINAL_GROUPS:
    longitudinal_result = run_longitudinal_comparison(
        LONGITUDINAL_GROUPS,
        metadata=LONGITUDINAL_METADATA,
        align=ALIGN,
        output_dir=LONGITUDINAL_OUTPUT if ALIGN else None,
    )
    display(longitudinal_result.validation.to_frame())
    if not longitudinal_result.validation.has_errors:
        display(longitudinal_result.group_day_summary)
        save_comparison_tables(
            longitudinal_result.tables,
            LONGITUDINAL_OUTPUT / 'longitudinal_comparison.xlsx',
        )
else:
    print('Configure LONGITUDINAL_GROUPS to run this section.')

Configure LONGITUDINAL_GROUPS to run this section.


## Treatment groups

Set the independent experimental unit for the current experiment. Common choices are `well`, `region`, or `animal`. Longitudinal-within-treatment analysis is optional; alignment is only used when longitudinal analysis is enabled.

In [22]:
TREATMENT_GROUPS = {
    'TSP-ONC' : r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_TSP-1_ONC_Cohort 1",
    'TSP-CTRL' : r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_TSP-1_CTRL_Cohort 1",
    'INS-ONC' : r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_INS_ONC_Cohort 1",
    'INS-CTRL' : r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_INS_CTRL_Cohort 1",
    'ChABC-CTRL' : r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1",
    'ChABC-ONC' : r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_ONC_Cohort 1",
    # 'control': Path(r'C:\\path\\to\\control'),
    # 'drug': Path(r'C:\\path\\to\\drug'),
}
TREATMENT_METADATA = {
    'video': 'region',
    'video_parent' : 'drug-treatment',
    'video_grandparent': 'animal',
    'video_ancestor_3': 'injury-treatment',
}
REPLICATE_UNIT = 'region'  # well, region, animal, or another metadata column
LONGITUDINAL_WITHIN_TREATMENT = False
ALIGN_WITHIN_TREATMENT = False
TREATMENT_OUTPUT = PROJECT_ROOT / 'comparison_outputs' / 'treatments'

In [23]:
treatment_result = None
if TREATMENT_GROUPS:
    treatment_result = run_treatment_comparison(
        TREATMENT_GROUPS,
        replicate_unit=REPLICATE_UNIT,
        metadata=TREATMENT_METADATA,
        longitudinal=LONGITUDINAL_WITHIN_TREATMENT,
        align=ALIGN_WITHIN_TREATMENT,
        output_dir=TREATMENT_OUTPUT if ALIGN_WITHIN_TREATMENT else None,
    )
    display(treatment_result.validation.to_frame())
    if not treatment_result.validation.has_errors:
        display(treatment_result.treatment_summary)
        if LONGITUDINAL_WITHIN_TREATMENT:
            display(treatment_result.treatment_day_summary)
        save_comparison_tables(
            treatment_result.tables,
            TREATMENT_OUTPUT / 'treatment_comparison.xlsx',
        )
else:
    print('Configure TREATMENT_GROUPS to run this section.')

,severity,code,group,video_path,message
0,error,missing_video_path_column,,,Explicit metadata requires a video_path column.
1,error,missing_metadata,TSP-ONC,C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo...,Required metadata field 'region' is missing.
2,error,missing_metadata,TSP-ONC,C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo...,Required metadata field 'region' is missing.
3,error,missing_metadata,TSP-ONC,C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo...,Required metadata field 'region' is missing.
4,error,missing_metadata,TSP-ONC,C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo...,Required metadata field 'region' is missing.
...,...,...,...,...,...
100,warning,low_replicate_count,TSP-CTRL,,Treatment 'TSP-CTRL' has only 1 independent re...
101,warning,low_replicate_count,INS-ONC,,Treatment 'INS-ONC' has only 1 independent reg...
102,warning,low_replicate_count,INS-CTRL,,Treatment 'INS-CTRL' has only 1 independent re...
103,warning,low_replicate_count,ChABC-CTRL,,Treatment 'ChABC-CTRL' has only 1 independent ...


## Generic filesystem sibling comparisons

This retains the original arbitrary hierarchy behavior: immediate child folders are compared at every internal node, using persisted per-video summaries.

In [24]:
HIERARCHICAL_ROOT = Path(r"C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710")  # e.g. Path(r'C:\\path\\to\\experiment')

if HIERARCHICAL_ROOT is not None:
    hierarchy_result = run_hierarchical_comparison(HIERARCHICAL_ROOT)
    display(hierarchy_result.validation.to_frame())
    if not hierarchy_result.validation.has_errors and hierarchy_result.root is not None:
        save_comparisons(
            root=hierarchy_result.root,
            sibling_tables=hierarchy_result.sibling_tables,
        )
        for node_path, table in hierarchy_result.sibling_tables.items():
            print(f'\nNode: {node_path}')
            display(table)
else:
    print('Set HIERARCHICAL_ROOT to run generic sibling comparisons.')

,severity,code,group,video_path,message



Node: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710


,child,n_videos,n_neurons,frac_grouped,frac_ungrouped,n_groups_light-evoked,mean_group_size_light-evoked,median_group_size_light-evoked,mean_group_corr_light-evoked,mean_spikes_per_group_light-evoked,...,rise_slope_hz_within_grouped,rise_slope_hz_between_grouped,half_max_width_seconds_var_ungrouped,half_max_width_seconds_within_ungrouped,half_max_width_seconds_between_ungrouped,spike_frequency_var_grouped,spike_frequency_within_grouped,spike_frequency_between_grouped,n_cells_ON_1_response(s),n_cells_ON_2_response(s)
0,DoD_ChABC_CTRL_Cohort 1,11,1,0.000000,1.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DoD_ChABC_ONC_Cohort 1,2,0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DoD_INS_CTRL_Cohort 1,21,9,0.111111,0.888889,1.0,1.0,1.0,0.0,4.000000,...,0.000590,0.000000,28.969210,24.829682,4.139528,0.000000,0.000000,0.000000,NaN,NaN
3,DoD_INS_ONC_Cohort 1,17,6,0.500000,0.500000,3.0,1.0,1.0,0.0,3.000000,...,0.002767,0.000012,0.027382,0.027382,0.000000,0.000147,0.000039,0.000108,1.000000,NaN
4,DoD_TSP-1_CTRL_Cohort 1,18,24,0.541667,0.458333,2.0,6.5,6.5,0.0,13.500000,...,0.003719,0.000000,19.283544,15.728915,3.554629,0.000391,0.000391,0.000000,6.500000,NaN
5,DoD_TSP-1_ONC_Cohort 1,29,44,0.477273,0.522727,6.0,3.5,3.5,0.0,8.833333,...,0.055343,0.000000,32.947912,32.736560,0.211352,0.000288,0.000288,0.000000,5.888889,2.333333



Node: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_ChABC_CTRL_Cohort 1


,child,n_videos,n_neurons,frac_grouped,frac_ungrouped,decay_tau_seconds_mean_unweighted,rise_slope_hz_mean_unweighted,decay_tau_seconds_mean_weighted,rise_slope_hz_mean_weighted,decay_tau_seconds_mean_ungrouped,...,rise_slope_hz_between_ungrouped,spike_frequency_var_unweighted,spike_frequency_within_unweighted,spike_frequency_between_unweighted,spike_frequency_var_weighted,spike_frequency_within_weighted,spike_frequency_between_weighted,spike_frequency_var_ungrouped,spike_frequency_within_ungrouped,spike_frequency_between_ungrouped
0,5965,2,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5968,1,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5971,4,1,0.0,1.0,15.338245,0.069073,15.338245,0.069073,15.338245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5972,4,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


...

,child,n_videos,n_neurons,frac_grouped,frac_ungrouped
0,5837L-1,1,0,0.0,0.0
1,5837L-2,1,0,0.0,0.0
2,5837L-3,1,0,0.0,0.0



Node: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_TSP-1_ONC_Cohort 1\5839\TSP-1


,child,n_videos,n_neurons,frac_grouped,frac_ungrouped,decay_tau_seconds_mean_unweighted,rise_slope_hz_mean_unweighted,decay_tau_seconds_mean_weighted,rise_slope_hz_mean_weighted,decay_tau_seconds_mean_ungrouped,...,rise_slope_hz_between_ungrouped,spike_frequency_var_unweighted,spike_frequency_within_unweighted,spike_frequency_between_unweighted,spike_frequency_var_weighted,spike_frequency_within_weighted,spike_frequency_between_weighted,spike_frequency_var_ungrouped,spike_frequency_within_ungrouped,spike_frequency_between_ungrouped
0,5839R-1,1,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5839R-2,1,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5839R-3,1,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5839R-4,1,1,0.0,1.0,4.881297,0.128357,4.881297,0.128357,4.881297,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Node: C:\Users\mzinn1\Desktop\DoD WT Cohorts\Ex Vivo Ca Imaging 710\DoD_TSP-1_ONC_Cohort 1\6040\TSP-1


,child,n_videos,n_neurons,frac_grouped,frac_ungrouped
0,6040R-1,1,0,0.0,0.0
1,6040R-2,1,0,0.0,0.0
